In [0]:
# ================================================
# READ FROM VALIDATED OUTPUT
# ================================================
from pyspark.sql.functions import col, count, sum as spark_sum

df_final_clean = spark.table(
    "default.nottinghamshire_validated"
)

print(f"✅ Rows loaded: {df_final_clean.count():,}")
print(f"✅ Columns: {df_final_clean.columns}")

In [0]:
# ================================================
# AGGREGATION LAYER
# Grain: lsoa_code × lsoa_name × year × 
#        month_num × month_name × crime_type
# ================================================

df_aggregated = df_final_clean.groupBy(
    "lsoa_code",
    "lsoa_name",
    "year",
    "month_num",
    "month_name",
    "crime_type"
).agg(
    count("*").alias("total_crimes")
).orderBy(
    "year",
    "month_num",
    "lsoa_code",
    "crime_type"
)

print(f"✅ Aggregated rows: {df_aggregated.count():,}")
display(df_aggregated.limit(20))

In [0]:
# ================================================
# VALIDATION
# ================================================

print("=== GRAIN DUPLICATE CHECK ===")
grain_dupes = df_aggregated.groupBy(
    "lsoa_code", "year",
    "month_num", "crime_type"
).count() \
.filter(col("count") > 1)
print(f"Duplicate rows at grain: {grain_dupes.count()}")

print("\n=== TOTAL CRIMES RECONCILIATION ===")
total = df_aggregated.agg(
    spark_sum("total_crimes")
).collect()[0][0]
print(f"Sum of total_crimes: {total:,}")
print(f"Expected: 134,927")
print(f"Match: {'✅' if total == 134927 else '⚠️'}")

print("\n=== NULL CHECK ===")
for field in df_aggregated.columns:
    null_count = df_aggregated.filter(
        col(field).isNull()
    ).count()
    status = "✅" if null_count == 0 else "⚠️"
    print(f"{status} {field}: {null_count} nulls")

In [0]:
# ================================================
# SAVE
# ================================================

df_aggregated.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.nottinghamshire_reporting")

print("✅ Saved to default.nottinghamshire_reporting")
print(f"✅ Total rows: {df_aggregated.count():,}")
print(f"✅ Columns: {df_aggregated.columns}")